# 1. Introducción.


Dado un conjunto finito $E$, se denota $2^E$ como el conjunto de sus partes. A efectos de la programación, se relacionará $E$ con $\{1, \dots, n\}$. A este conjunto lo denotaremos como $[n]$. Los elementos de $E$ se denotarán con letras minúsculas, y los subconjuntos de $E$, con mayúsculas.

Se llama función de conjuntos a una función $f:2^E \to \mathbb{R}$. Si $i \in E$ y $A \subset E$, se usará la notación $f(i|A) = f(\{i\}\cup A) - f(A)$. Este valor se llama rendimiento marginal.

Se dice que $f$ es monótona creciente si $f(A) \leq f(B)$ siempre que $A \subset B$. Equivalentemente, es marginal si $f(i|A) \geq 0$ para cualesquiera $i, A$.

Se dice que $f$ es submodular si verifica
$$f(A \cup B) + f(A \cap B) \leq f(A) + f(B)$$
para cualesquiera $A, B \subset E$.
Las siguientes condiciones son equivalentes a la submodularidad.
- Para todo $A \subset B$ e $i \notin B$ se cumple $f(i|B) \leq f(i|A)$.
- Para todo $A$ e $i, j \notin A$ se cumple $f(i,j|A) \leq f(i|A) + f(j|A)$.
Por tanto, la submodularidad se conoce como *propiedad de rendimientos decrecientes*.

# 2. El algoritmo greedy

## 2.1. Introducción

Se buscan algoritmos para resolver problemas del tipo
$$ \max \{f(A) : A \subset E, |A| = k\},$$
donde $f$ es submodular. Este problema es NP-difícil en general, así que se requieren algoritmos aproximados. El algoritmo más usual para resolverlo es el algoritmo *greedy* (voraz o míope).
El algoritmo *greedy* consiste en los siguientes pasos:
```
    A(0) = {}
    l = 1       ###  l es el contador
    while (l <= k)
        i(l) = arg max{f(i|A(l-1)) : i \notin A(l-1)}
        A(l) = A(l-1) + i(l)
        l = l+1
```
Este algoritmo no requiere la submodularidad de $f$. De hecho, se usa constantemente de forma heurística, en problemas de elección que aparecen en Estadística, *Machine learing*, etc. La submodularidad de $f$ aporta cotas robustas a la solución *greedy*, así como mejoras computacionales al método.

El siguiente bloque de código contiene una implementación sencilla del algoritmo greedy.

In [ ]:
import sys

def greedy(f, n, k):
    """
    f - Funcion
    n - Cardinal del conjunto base
    k - Maxima longitud de la lista
    """
    indices = []
    valores = []
    cnt = 0
    while cnt < k:
        cnt = cnt+1
        max_valor = -sys.float_info.max # Cota minima
        max_indice = -1 # Valor nul
        for i in range(n):
            valor_greedy = -sys.float_info.max
            if (i not in indices):
                valor_greedy = f(indices + [i])
                if valor_greedy > max_valor:
                    # En este caso mi valor es mejor:
                    max_indice = i
                    max_valor = valor_greedy
        indices.append(max_indice)
        valores.append(max_valor)
    return indices, valores


## 2.2. La cota de Nemhauser

Cuando la función $f$, además de ser submodular, es monótona creciente y $f(\emptyset) = 0$, esta goza de propiedades adicionales. En particular, se tiene una cota robusta para la solución *greedy*.

**Teorema** (Nemhauser).
Sea $f$ submodular creciente con $f(\emptyset)=0$. Sea $A^*$ la solución óptima del problema $\max \{f(A) : |A| =k\}$ y sea $A_k$ la solución dada por el algoritmo *greedy*. Entonces
$$f(A_k) \geq \left( 1- \frac{1}{e} \right)f(A^*).$$

En la práctica, la solución *greedy* suele ser mucho mejor que la cota que nos proporciona este teorema. Sin embargo, esta cota es óptima, (no exuste una cota universal más alta).

## 2.3. El algoritmo *lazy greedy*.

Analizando el algoritmo *greedy*, vemos que en cada iteración se calculan todos los $f(i|S_k)$ se calcula su máximo. Pero sabemos que $f(i|S_{k-1}) \geq f(i|S_k)$.
Si se da el caso en el que hemos calculado $f(i|S_k)$, y aún no hemos calculado $f(j|S_k)$, pero sabemos que $f(i|S_k) \geq f(j|S_{k-1})$; entonces se deduce que $f(i|S_k) \geq f(j|S_k)$, por lo que no puede ser máximo, y no es necesario calcularlo.
Este hecho permite redefinir el algoritmo *greedy* para ahorrar evaluaciones de $f$, obteniendo el mismo resultado. El algoritmo resultante es el llamado *lazy greedy*. Es más complejo, puesto que requiere una cola de prioridad, pero reduce muchísimas operaciones.

El algoritmo en pseudo código se expresa a continuacion.
```
Asignar S(0) = {}.
Asignar G como un array de dimensión n.
Para cada i=1,...,n, Asignar G(i) = f({i}).
Para cada k = 1,...,m:
    Bucle (b):
        Asignar i = arg max {G(i): i \notin S(k-1)}.
        Si f(i|S(k-1)) no se ha calculado.
            Asignar G(i) = f(i|S(k-1)).
            Repetir bucle (b).
        Si f(i|S(k-1)) sí se ha calculado:
            Asignar i(k) = i.
            Asignar S(k) = S(k-1) \cup {i(k)}$. 
            Salir del bucle (b).
```

Se ha usado una cola de prioridad (*priority queue* o *heap queue*), que es una estructura de datos que permite añadir valores arbitrarios y eliminar el más grande con eficiencia logarítmica (requiere el paquete `headpq`). La siguiente celda contiene la implementación en Python del *lazy greedy*.


In [ ]:
import heapq

def lazy_greedy(f, n, m):
    A = [] # vector de indices
    f_A = [] # vector de 
    orden = list(range(n))
    f_A_k = f([]) # El último f_A calculado
    ganancia = [(-f([i]) + f_A_k, i) for i in range(n)]
    heapq.heapify(ganancia)

    # ganancia - array de ganancias. Es un array cuyos componentes son duplas (x,i) donde i es el índice.
    # $x=f(i|S_{k})$ cuando se calculó por ultima vez.
    # Se ha convertido en una cola de prioridad (max-heap) (paquete heapq), con el fin de hacer más eficientes la
    # inserción y extracción del valor de maxima prioridad.
    actualizado = list(range(n))
    for k in range(m):
        while True:
            gan0, i0 = heapq.heappop(ganancia)
            if i0 in actualizado:
                break # Este es el i0 que buscamos
            else:
                actualizado.append(i0)
                gan0 = -(f(A+[i0]) - f_A_k)
                heapq.heappush(ganancia, (gan0, i0))
            # Se ha cogido el indice optimo. Actualizo lista.
        A.append(i0)
        f_A.append(-gan0 + f_A_k)

        # Preparo todo para el próximo bucle
        f_A_k = f_A[-1]
        actualizado.clear()
            
    return A, f_A
        


# 3. Aplicaciones a la estadística

In [ ]:
def log_det(seleccion, V):
    """ V: Matriz de covarianzas """
    V_seleccion = np.take(np.take(V, seleccion, axis=0), seleccion, axis=1)
    return np.log(np.linalg.det((V_seleccion)))


In [ ]:
def log_det_obs(selección, V):
    return np.log(np.linalg.det(sum(V[k] for k in seleccion)))

# 4. Implementación. El Wine Dataset.

Veremos aplicaciones de la submodularidad a una serie de problemas estadísticos clásicos.
Usaremos el Wine dataset 

## 4.1. Estandarización de datos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
CSV_PATH = "wine-quality/winequality-red.csv"

df_bruta = pd.read_csv(CSV_PATH, sep=';', decimal=',', engine='python').drop(columns=['quality'])

print(df_bruta.describe())
df_zscore = (df_bruta-df_bruta.mean())/df_bruta.std()
X = df_zscore.to_numpy()
n = len(df_zscore)



## 4.2. Selección de variables.

In [ ]:
# 2. Calculo matriz de correlaciones
matriz_corr = df_zscore.corr()

plt.figure(figsize=(10, 8))

# Creamos el mapa de calor
sns.heatmap(matriz_corr, cmap='coolwarm', fmt=".2f")

plt.title("Matriz de Correlación")


A continuación, aplicamos el algoritmo *greedy* para hallar las observaciones más significativas.

In [ ]:
np_corr = matriz_corr.to_numpy()
variables = matriz_corr.columns.tolist()
lista_greedy = greedy(lambda S: log_det(S, np_corr), 11, 11)
variables_ordenadas = [variables[i] for i in lista_greedy[0]]

resultados = [(i+1, variables_ordenadas[i], lista_greedy[1][i]) for i in range(11)]
df_orden_variables = pd.DataFrame(resultados, columns=['Puesto', 'Variable', 'Log-det'])
print(df_orden_variables)

In [ ]:
print(df_orden_variables.to_latex(index=False, 
                        formatters={'Log-det': "{:,.4f}".format}, # 4 decimales
                        column_format='clr', # c=center, l=left, r=right
                        caption="Progreso del log-determinante según la variable seleccionada."))

## 4.3. Selección de observaciones

In [ ]:
# 1. Añadimos una columna de unos a la matriz X.
X_aum = np.column_stack((np.ones(n), X))
xxt = [np.outer(X_aum[i,:], X_aum[i,:]) for i in range(n)] # Productos externos
observaciones_greedy = lazy_greedy(lambda S: np.log(np.linalg.det(np.eye(12)+sum(xxt[i] for i in S))), n, 20)
print(observaciones_greedy)

La siguiente tabla muestra el valor del log determinante en cada iteración (se recogen las 20 primeras).

In [ ]:
df_observaciones = pd.DataFrame({
    "Obervación": [0] + observaciones_greedy[0],
    "Log_determinante": [0] + observaciones_greedy[1],
})
df_observaciones

A continuación, hago un plot de 

### Comparativa greedy - lazy greedy

1. Comparación en tiempo.

In [ ]:
from time import perf_counter
t0 = perf_counter()
greedy(lambda S: np.log(np.linalg.det(np.eye(12)+sum(xxt[i] for i in S))), n, 100)
t1 = perf_counter()
print("Tiempo del algoritmo greedy:", t1-t0)
t0 = perf_counter()
lazy_greedy(lambda S: np.log(np.linalg.det(np.eye(12)+sum(xxt[i] for i in S))), n, 100)
t1 = perf_counter()
print("Tiempo del algoritmo lazy greedy:", t1-t0)

2. Comparación en número de ejecuciones.